<a href="https://colab.research.google.com/github/EigenFunction271/lora_colab_unsloth/blob/main/Colab_unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Install dependencies

This section installs all the libraries needed for modern LoRA fine-tuning.

_unsloth_ is the optimization layer. It rewrites parts of the training process using highly optimized GPU kernels so that training uses less memory and runs significantly faster. This is what makes it possible to fine-tune reasonably capable models on free Google Colab GPUs.

_transformers_ is Hugging Face’s core library for loading and interacting with large language models. It handles tokenization, model loading, inference, and generation.

_datasets_ is Hugging Face’s dataset library. It provides easy access to datasets stored online and efficient tools for preprocessing them.

_peft_ stands for Parameter Efficient Fine Tuning. This library implements techniques like LoRA and QLoRA, where only small portions of the model are trained instead of the full neural network.

_trl_ contains training utilities specialized for language models. In this notebook, we use its SFTTrainer class for supervised fine-tuning.

_bitsandbytes_ enables 4-bit quantization. Normally, model weights are stored in 16-bit or 32-bit precision, which consumes a huge amount of GPU memory. Quantization compresses these weights so the model can fit on consumer hardware.

In [ ]:
!pip install unsloth
!pip install --no-deps transformers accelerate peft trl datasets bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

# 2. Load model

We load a small instruction-tuned language model using 4-bit quantization. Instead of loading the model in full precision, QLoRA compresses the frozen base weights to reduce memory usage dramatically while preserving most performance. This allows us to fine-tune models that would normally exceed the memory limits of consumer hardware. The tokenizer converts text into token IDs the model can process.

_FastLanguageModel.from_pretrained(...)_ loads a pretrained language model and tokenizer.

_model_name_ specifies which base model to download. We use a small instruction-tuned Qwen model because it is lightweight enough for Colab while still producing good outputs.

_load_in_4bit = True_ enables QLoRA-style quantization. The model weights are compressed into 4-bit precision to dramatically reduce memory usage.

The function returns:
* model: the neural network itself
* tokenizer: the component that converts text into tokens the model can process

In [ ]:
import unsloth
from unsloth import FastLanguageModel #imports Unsloth’s optimized wrapper around Hugging Face models. This version patches the model internally for faster training and lower memory usage.
import torch #imports PyTorch, the deep learning framework used underneath almost all modern LLM tooling. PyTorch handles tensors, gradients, GPU execution, and backpropagation.

max_seq_length = 2048 #sets the maximum number of tokens the model can process at once. Longer sequence lengths allow more context but consume more VRAM.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


We use:
- a small instruct model
- 4-bit quantization (QLoRA)
- consumer GPU friendly setup

# 3. Add LoRA adapters

This section prepares the model for LoRA fine-tuning.

_get_peft_model(...)_ modifies the existing model by inserting small trainable matrices called LoRA adapters into specific parts of the transformer network. The original model weights remain frozen, only these small inserted matrices are trained.

_r = 16_ sets the LoRA rank. This controls the size of the trainable adapter matrices. Higher ranks increase model capacity but require more memory and computation.

_target_modules_ specifies which parts of the transformer should receive LoRA adapters. These are mostly attention and feed-forward projection layers inside the transformer blocks.

For example:

* _q_proj, k_proj, v_proj_ correspond to query, key, and value projections in self-attention

* _o_proj_ is the attention output projection

* _up_proj_ and _down_proj_ are feed-forward network projections

These layers are where much of the model’s behavior emerges, making them good targets for efficient adaptation.

_lora_alpha_ controls the scaling strength of the LoRA updates.

_lora_dropout = 0_ disables dropout for simplicity.

_bias = "none"_ means we are not training bias parameters.

_use_gradient_checkpointing = "unsloth"_ reduces VRAM usage by recomputing some activations during backpropagation instead of storing all of them in memory.

Conceptually, this entire block says:
“Insert small trainable correction matrices into important parts of the transformer while keeping the original brain mostly frozen.”

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

Unsloth 2026.5.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


# 4. Load Shakespeare dataset

The dataset contains pairs of:
* modern English
* Shakespearean English

This gives us supervised training examples:
* input sentence
* desired output sentence

We intentionally use a small dataset because this is a teaching demo, not a production training run.

Even with only a few hundred examples, LoRA can noticeably shift the model’s behavior.

By repeatedly showing the model examples of how sentences should be transformed, we gradually shift the model’s output distribution toward the desired speaking style.

In [ ]:
from datasets import load_dataset #downloads a dataset directly from Hugging Face.

dataset = load_dataset(
    "harpreetsahota/modern-to-shakesperean-translation",
    split="train" #loads the training split of the dataset.
)

dataset = dataset.select(range(200)) #keeps only the first 200 examples.

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/22.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/274 [00:00<?, ? examples/s]

# 5. Inspect examples

This section inspects the dataset structure before preprocessing. Fine-tuning datasets are fundamentally just structured input-output pairs.

Before training, we need to verify:
* which columns contain the source text
* which columns contain the target output
* whether the data quality looks reasonable

The dataset uses:
* modern
* shakespearean

as column names.

In [ ]:
print(dataset.column_names) #shows the available fields in each training example.
print(dataset[0]) #prints the first row of the dataset.

['modern', 'shakespearean']
{'modern': "When someone says 'She's thirsty, ain't she?', they're implying she's seeking attention.", 'shakespearean': 'When one remarks, "She doth crave attention, doth she not?", they suggest her desire for notice.'}


# 6. Convert into text format

Most modern instruction-tuned models expect data in an instruction-response format. This section converts the raw dataset into instruction-following prompts.

* example['modern'] accesses the modern English sentence.
* example['shakespearean'] accesses the target Shakespearean translation.

The formatted text combines:
* an instruction
* an input
* a desired response

This mirrors how modern instruction-tuned chat models are trained.
* return {"text": text} creates a new field called "text" containing the formatted prompt.
* dataset.map(...) applies this formatting function to every row in the dataset.

Conceptually, we are teaching the model:
“When you see this kind of instruction, produce outputs in this style.”

In [ ]:
def formatting_prompts_func(example): #defines a Python function that transforms each dataset row into a formatted training example.
    text = f"""### Instruction:
Convert the following sentence into Shakespearean English.

### Input:
{example['modern']}

### Response:
{example['shakespearean']}"""

    return {"text": text}

dataset = dataset.map(formatting_prompts_func)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

# 7. Show BEFORE fine-tuning

This section tests the model before training.

* _prompt_ contains the input text we want the model to respond to.

This gives us a baseline to compare against after fine-tuning.

In [ ]:
FastLanguageModel.for_inference(model) #switches the model into inference mode for faster generation.

prompt = """
Convert the following sentence into Shakespearean English.

Modern English:
Bro I need coffee immediately.

Shakespearean:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda") #tokenizer converts the prompt into tensors the neural network can process, .to("cuda") moves the tensors onto the GPU.

outputs = model.generate( #Performs autoregressive text generation. The model predicts tokens one at a time based on previous tokens.
    **inputs,
    max_new_tokens=64, #limits how many new tokens the model can generate.
)

#tokenizer.decode(...) converts generated token IDs back into readable text.

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/


Convert the following sentence into Shakespearean English.

Modern English:
Bro I need coffee immediately.

Shakespearean:
Ay, my dearest friend, it is now time for thee to partake in a cup of strong and invigorating liquid. Hast thou not heard the sweet whisperings of the siren song of caffeine? Let us then set our cups upon the table, and let the battle commence!

The translation aims to capture


# 8. Train

This is the actual fine-tuning phase. For each example, the model generates predictions, computes a loss measuring how different its output is from the target response, and backpropagates gradients through the network. However, because the base weights are frozen, only the LoRA adapter weights are updated. Over many iterations, these small matrices gradually learn how to steer the model toward Shakespearean-style outputs.

In [ ]:
!pip install trl

In [ ]:
from trl import SFTTrainer #SFTTrainer is a supervised fine-tuning trainer specialized for language models.
from transformers import TrainingArguments #TrainingArguments stores all training hyperparameters.

trainer = SFTTrainer( # This initializes the training pipeline.
    model = model, # The LoRA-modified language model.
    tokenizer = tokenizer, # handles text preprocessing
    train_dataset = dataset, # the formatted dataset.
    dataset_text_field = "text", # Tells the trainer which dataset field contains the actual training prompts
    max_seq_length = max_seq_length, # Sets the maximum sequence length during training.
    args = TrainingArguments( # defines training hyperparameters.
        per_device_train_batch_size = 2, # How many examples are processed simultaneously on the GPU.
        gradient_accumulation_steps = 4, # Accumulates gradients across multiple mini-batches before updating weights. This simulates a larger batch size while staying within VRAM limits.
        warmup_steps = 5, # Gradually increases the learning rate at the start of training for stability.
        max_steps = 60, # Total number of optimization steps.
        learning_rate = 2e-4, # Controls how aggressively weights are updated during gradient descent.
        logging_steps = 1,
        optim = "adamw_8bit", # Uses an 8-bit memory-efficient AdamW optimizer.
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        fp16 = not torch.cuda.is_bf16_supported(), #Chooses mixed precision settings depending on GPU support.
        bf16 = torch.cuda.is_bf16_supported(), # Mixed precision speeds up training and reduces memory usage.
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.446002
2,3.506201
3,3.256117
4,2.806050
5,2.671179
6,2.472547
7,2.236509
8,2.009650
9,2.050598
10,1.778434


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


TrainOutput(global_step=60, training_loss=1.6913894176483155, metrics={'train_runtime': 122.2371, 'train_samples_per_second': 3.927, 'train_steps_per_second': 0.491, 'total_flos': 190167994970112.0, 'train_loss': 1.6913894176483155, 'epoch': 2.4})

# 9. Show AFTER fine-tuning



In [ ]:
FastLanguageModel.for_inference(model)

prompt = """
Convert the following sentence into Shakespearean English.

Modern English:
Bro I need coffee immediately.

Shakespearean:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=64,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



Convert the following sentence into Shakespearean English.

Modern English:
Bro I need coffee immediately.

Shakespearean:
My dear friend, haste thou to procure me a cup of strong ale at once. 

Note: In Shakespearean English, "bro" is used as an informal term of endearment for friends. Also, "coffee" is considered vulgar in Shakespearean times and would be replaced with "ale," which was more


# 10. Save adapter

We save the LoRA adapter weights separately from the original model. Since LoRA only trains small correction matrices, the resulting files are dramatically smaller than the full model weights. This makes LoRA adapters cheap to distribute, store, and swap between tasks. The original base model remains unchanged, while different adapters can specialize it for different behaviors or domains.


In [ ]:
model.save_pretrained("shakespeare_lora")
tokenizer.save_pretrained("shakespeare_lora")

Unsloth: Restored added_tokens_decoder metadata in shakespeare_lora/tokenizer_config.json.


('shakespeare_lora/tokenizer_config.json',
 'shakespeare_lora/chat_template.jinja',
 'shakespeare_lora/tokenizer.json')